In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm
import os
import sys
import argparse
from pathlib import Path

cwd = Path.cwd()
print("Current working directory:", cwd) # /path/to/home
root_path = Path("/path/to/BrainWear_Kareem")
# root_path = Path("/path/to/BrainWear_Kareem")
project_root = root_path/"FYP"

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f"Added to path: {project_root}")

from datasets.brats2020 import BraTSDataset, batch_seg_to_slot_targets
from baseline.autoencoder.models import ResNetSpatialEncoder, ResNetAutoencoder, ResNetClassifier

DEFAULT_DATA_DIR = root_path/"Processed_BraTS2020_TrainingData"
SAVE_DIR = project_root/"baseline"/"autoencoder"/"models"/"encoder"

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("--data_dir", type=str, default=DEFAULT_DATA_DIR)
parser.add_argument("--batch_size", type=int, default=4)
parser.add_argument("--epochs", type=int, default=50)
parser.add_argument("--lr", type=float, default=1e-4)
parser.add_argument("--weight_decay", type=float, default=1e-5)
parser.add_argument("--max_patients", type=int, default=None)
parser.add_argument("--model_name", type=str, default="resnet18", choices=["resnet18", "resnet50"])
parser.add_argument("--patience", type=int, default=10, help="Epochs to wait for val loss improvement before stopping")
args = parser.parse_args([
    '--model_name', 'resnet18',
    '--lr', '0.0004',
    '--epochs', '150',
    '--patience', '15',
])

save_name = f"{args.model_name}_{args.epochs}e_{args.lr}lr.pt"
save_location = SAVE_DIR/save_name

print("Initialising Dataset")
dataset = BraTSDataset(
    root_dir=DEFAULT_DATA_DIR, 
    target_size=(64, 96, 96), 
    shuffle=True,
    max_patients=args.max_patients
)

# 80/20 Train/Validation Split
total_size = len(dataset)
val_size = int(0.2 * total_size)
train_size = total_size - val_size

# Using PyTorch's built-in random splitter
train_dataset, val_dataset = random_split(
    dataset, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset, batch_size=args.batch_size, shuffle=True, 
    num_workers=4, pin_memory=True
)

# Don't shuffle eval data
val_loader = DataLoader(
    val_dataset, batch_size=args.batch_size, shuffle=False, 
    num_workers=4, pin_memory=True
)

print(f"Total Patients: {total_size} | Training: {train_size} | Validation: {val_size}")

model = ResNetAutoencoder(model_name=args.model_name).to(device)

recon_error = nn.MSELoss()

optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

print(f"Starting training using {args.model_name}")
model.train()

best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(args.epochs):
    # Training
    model.train()
    train_loss = 0.0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{args.epochs} [Train]", leave=False)
    
    for batch_idx, (t2_imgs, seg_masks, scores) in enumerate(progress_bar):
        t2_imgs = t2_imgs.to(device)
        
        optimizer.zero_grad()
        x_hat, _ = model(t2_imgs)
        loss = recon_error(x_hat, t2_imgs)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        progress_bar.set_postfix({'MSE': f"{loss.item():.4f}"})
        
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for t2_imgs, seg_masks, scores in val_loader:
            t2_imgs = t2_imgs.to(device)
            
            x_hat, _ = model(t2_imgs)
            loss = recon_error(x_hat, t2_imgs)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    
    # Early stopping
    print(f"Epoch [{epoch+1}/{args.epochs}] | Train MSE: {avg_train_loss:.4f} | Val MSE: {avg_val_loss:.4f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        
        # Only save the model if it actually improved
        torch.save(model.state_dict(), save_location)
        print(f"Val Loss improved. Model saved to {save_location}")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s)")
        
        if epochs_no_improve >= args.patience:
            print(f"Early stopping triggered after {epoch+1} epochs. Best Val MSE: {best_val_loss:.4f}")
            break # Exit the training loop entirely

print("Training Complete")

In [ ]:
import matplotlib.pyplot as plt

def visualise_reconstructions(
    ae_weights_path: str,
    brats_root: str,
    model_name: str = "resnet18",
    num_samples: int = 4,
    save_path: str = "reconstructions.png",
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
    save_fig: bool = True,
    shuffle: bool = False,
):
    device = torch.device(device)

    model = ResNetAutoencoder(model_name=model_name)
    state = torch.load(ae_weights_path, map_location=device)
    if "model_state_dict" in state:
        state = state["model_state_dict"]
    model.load_state_dict(state)
    model.eval().to(device)

    dataset = BraTSDataset(root_dir=brats_root, target_size=(64, 96, 96))
    loader  = DataLoader(dataset, batch_size=1, shuffle=shuffle)

    samples = []
    with torch.no_grad():
        for t2_imgs, seg_imgs, labels in loader:
            vol = t2_imgs.to(device)
            recon, _ = model(vol)
            samples.append((
                vol.cpu().squeeze(0).squeeze(0),    # (D, H, W)
                recon.cpu().squeeze(0).squeeze(0),  # (D, H, W)
            ))
            if len(samples) == num_samples:
                break

    n_rows = 2
    n_cols = num_samples * 2
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.5))
    fig.suptitle("Autoencoder Reconstructions  |  odd cols = input, even cols = reconstruction", fontsize=10)

    for row_ax, label in zip(axes[:, 0], ["Axial (mid-depth)", "Diff |input - recon|"]):
        row_ax.set_ylabel(label, fontsize=8)

    for s_idx, (orig, recon) in enumerate(samples):
        D        = orig.shape[0]
        col_orig  = s_idx * 2
        col_recon = s_idx * 2 + 1

        orig_sl  = orig [D // 2, :, :]
        recon_sl = recon[D // 2, :, :]
        vmin, vmax = orig_sl.min().item(), orig_sl.max().item()

        # Row 0: axial slice
        axes[0, col_orig ].imshow(orig_sl,  cmap="gray", vmin=vmin, vmax=vmax, aspect="auto")
        axes[0, col_recon].imshow(recon_sl, cmap="gray", vmin=vmin, vmax=vmax, aspect="auto")
        axes[0, col_orig ].set_title(f"S{s_idx+1} input", fontsize=7)
        axes[0, col_recon].set_title(f"S{s_idx+1} recon", fontsize=7)

        # Row 1: difference map
        diff = (orig_sl - recon_sl).abs()
        im = axes[1, col_orig].imshow(diff, cmap="hot", aspect="auto")
        axes[1, col_orig ].set_title(f"S{s_idx+1} diff",  fontsize=7)
        plt.colorbar(im, ax=axes[1, col_orig], fraction=0.046)
        axes[1, col_recon].axis("off")

    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    if save_fig:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    # plt.close()
    print(f"Saved reconstruction grid at {save_path}")

weights_name = "resnet18_50e_0.0001lr"
model_name = weights_name.split("_")[0]
VISUALISATION_PATH = Path("/path/to/BrainWear_Kareem/FYP/baseline/autoencoder/models/visualisations")
SHUFFLE_SAMPLES = False
FOLDER_NAME = "random" if SHUFFLE_SAMPLES else "same"

visualise_reconstructions(
    ae_weights_path=f"/path/to/BrainWear_Kareem/FYP/baseline/autoencoder/models/encoder/{weights_name}.pt",
    brats_root=str(DEFAULT_DATA_DIR),
    model_name=model_name,
    num_samples=4,
    save_path=VISUALISATION_PATH/FOLDER_NAME/f"{weights_name}.png",
    save_fig=True,
    shuffle=SHUFFLE_SAMPLES
)